# Ejercicios 05 — Proyectos POO (Animales e Inmobiliaria)

**Teoría Tema 2:** clases, encapsulamiento, tipos de métodos, herencia y polimorfismo.

Este cuaderno es un **laboratorio condensado** inspirado en dos proyectos reales del curso:

| Proyecto | Repositorio | Pilares que practicas aquí |
|----------|-------------|----------------------------|
| Animales POO | [poo_animales](https://github.com/gracobjo/poo_animales) | Abstracción (`ABC`), herencia, encapsulamiento, polimorfismo |
| Sistema Inmobiliario | [poo_inmobiliaria](https://github.com/gracobjo/poo_inmobiliaria) | Properties, atributos de clase, enums, validación, colaboración entre objetos |

> Versión **didáctica y reducida**: no sustituye clonar y ejecutar los repos completos.
> Tras este notebook, abre los proyectos y compara con el código de producción.


---
# Parte A — Animales POO

Modelo: clase abstracta `Animal` + especies (`Perro`, `Gato`) + funciones de gestión polimórficas.


## A1. Abstracción: clase base `Animal`

`Animal` hereda de `ABC` y **no se puede instanciar**. Obliga a implementar `hacer_sonido()` y `tipo_alimentacion()`.


In [1]:
from abc import ABC, abstractmethod


class Animal(ABC):
    EDAD_MINIMA = 0
    EDAD_MAXIMA = 30

    def __init__(self, nombre: str, edad: int):
        if not isinstance(nombre, str) or not nombre.strip():
            raise ValueError("El nombre debe ser un texto no vacío.")
        self.nombre = nombre.strip()
        self.edad = edad  # pasa por el setter

    @property
    def edad(self) -> int:
        return self._edad

    @edad.setter
    def edad(self, valor: int) -> None:
        if isinstance(valor, bool) or not isinstance(valor, int):
            raise ValueError("edad debe ser un entero.")
        if not (self.EDAD_MINIMA <= valor <= self.EDAD_MAXIMA):
            raise ValueError(f"edad fuera de rango [{self.EDAD_MINIMA}, {self.EDAD_MAXIMA}].")
        self._edad = valor

    def info_basica(self) -> str:
        return f"{self.nombre} es un {self.__class__.__name__} de {self.edad} años."

    @abstractmethod
    def hacer_sonido(self) -> str:
        ...

    @abstractmethod
    def tipo_alimentacion(self) -> str:
        ...


# Intentar crear Animal directo debe fallar
try:
    Animal("Fantasma", 1)
except TypeError as e:
    print("OK — no se puede instanciar Animal:", type(e).__name__)

OK — no se puede instanciar Animal: TypeError


## A2. Herencia y encapsulamiento: `Perro` y `Gato`

Atributos con doble guion bajo (`__peso`, `__energia`) → *name mangling*.  
Acceso solo vía `@property` con validación.


In [2]:
class Perro(Animal):
    ENERGIA_MAX = 100
    COSTO_JUGAR = 20

    def __init__(self, nombre, edad, peso, color, energia=100):
        super().__init__(nombre, edad)
        self.__vacunado = False
        self.peso = peso
        self.color = color
        self.energia = energia

    @property
    def peso(self):
        return self.__peso

    @peso.setter
    def peso(self, valor):
        if not isinstance(valor, (int, float)) or isinstance(valor, bool) or valor <= 0:
            raise ValueError("peso debe ser > 0")
        self.__peso = float(valor)

    @property
    def color(self):
        return self.__color

    @color.setter
    def color(self, valor):
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("color no vacío")
        self.__color = valor.strip()

    @property
    def energia(self):
        return self.__energia

    @energia.setter
    def energia(self, valor):
        if not isinstance(valor, (int, float)) or isinstance(valor, bool):
            raise ValueError("energia numérica")
        self.__energia = max(0, min(self.ENERGIA_MAX, float(valor)))

    @property
    def vacunado(self):
        return self.__vacunado

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Guau!"

    def tipo_alimentacion(self):
        return "omnívoro"

    def jugar(self):
        if self.energia < self.COSTO_JUGAR:
            raise ValueError(f"{self.nombre} no tiene energía para jugar")
        self.energia -= self.COSTO_JUGAR
        return f"{self.nombre} juega. Energía={self.energia}"

    def vacunar(self):
        if self.__vacunado:
            raise ValueError(f"{self.nombre} ya está vacunado")
        self.__vacunado = True
        return f"{self.nombre} vacunado"


class Gato(Animal):
    def __init__(self, nombre, edad, peso, color, energia=100):
        super().__init__(nombre, edad)
        self.__presas = 0
        self.peso = peso
        self.color = color
        self.energia = energia

    @property
    def peso(self):
        return self.__peso

    @peso.setter
    def peso(self, valor):
        if not isinstance(valor, (int, float)) or isinstance(valor, bool) or valor <= 0:
            raise ValueError("peso debe ser > 0")
        self.__peso = float(valor)

    @property
    def color(self):
        return self.__color

    @color.setter
    def color(self, valor):
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("color no vacío")
        self.__color = valor.strip()

    @property
    def energia(self):
        return self.__energia

    @energia.setter
    def energia(self, valor):
        if not isinstance(valor, (int, float)) or isinstance(valor, bool):
            raise ValueError("energia numérica")
        self.__energia = max(0, min(100.0, float(valor)))

    @property
    def presas_cazadas(self):
        return self.__presas

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Miau!"

    def tipo_alimentacion(self):
        return "carnívoro"

    def cazar(self):
        if self.energia < 25:
            raise ValueError(f"{self.nombre} demasiado cansado para cazar")
        self.energia -= 25
        self.__presas += 1
        return f"{self.nombre} caza. Presas={self.__presas}, energía={self.energia}"


rex = Perro("Rex", 5, 12.0, "marrón")
misi = Gato("Misi", 3, 4.5, "blanco")
print(rex.info_basica())
print(rex.hacer_sonido(), "|", rex.tipo_alimentacion())
print(misi.hacer_sonido(), "|", misi.tipo_alimentacion())
print(rex.jugar())
print(misi.cazar())
print("Name mangling (no acceso directo a __peso):", hasattr(rex, "__peso"), "→ getattr _Perro__peso:", getattr(rex, "_Perro__peso"))

Rex es un Perro de 5 años.
Rex dice: ¡Guau! | omnívoro
Misi dice: ¡Miau! | carnívoro
Rex juega. Energía=80.0
Misi caza. Presas=1, energía=75.0
Name mangling (no acceso directo a __peso): False → getattr _Perro__peso: 12.0


## A3. Polimorfismo: gestión sin preguntar la especie

Las funciones reciben `Animal` y delegan en el método de la clase real.


In [3]:
def emitir_sonido(animal: Animal) -> str:
    return animal.hacer_sonido()


def listar_info(animales: list[Animal]) -> list[str]:
    return [a.info_basica() for a in animales]


def total_peso(animales: list[Animal]) -> float:
    return sum(a.peso for a in animales)


refugio = [rex, misi, Perro("Laika", 2, 8.0, "negro"), Gato("Luna", 1, 3.2, "gris")]
for a in refugio:
    print(emitir_sonido(a))
print("---")
for linea in listar_info(refugio):
    print(linea)
print("Peso total:", round(total_peso(refugio), 2), "kg")

Rex dice: ¡Guau!
Misi dice: ¡Miau!
Laika dice: ¡Guau!
Luna dice: ¡Miau!
---
Rex es un Perro de 5 años.
Misi es un Gato de 3 años.
Laika es un Perro de 2 años.
Luna es un Gato de 1 años.
Peso total: 27.7 kg


### Tu turno A

**Ejercicio A1.** Crea la clase `Loro(Animal)` con frase aprendida (`aprender` / `hablar`), `volar` (consume energía) y propiedades encapsuladas.

**Ejercicio A2.** Añade el loro al refugio y comprueba que `emitir_sonido` y `listar_info` funcionan sin cambiar esas funciones.


In [4]:
# Escribe tu solución aquí (Loro + prueba con el refugio)


---
# Parte B — Sistema Inmobiliario POO

Modelo reducido: `EstadoCasa`, `Vivienda`, `Inquilino`, regla del **35 %**, impuesto **AITP** y un mini-servicio de cartera.


## B1. Enums y atributos de clase

Estados cerrados con `Enum`. Impuestos compartidos en la clase (`impuesto_aitp`).


In [5]:
from enum import Enum


class EstadoCasa(Enum):
    DISPONIBLE = "disponible"
    RESERVADA = "reservada"
    VENDIDA = "vendida"
    ALQUILADA = "alquilada"


class Vivienda:
    total_viviendas = 0
    impuesto_aitp = 0.10  # 10 % sobre el precio en la compra

    def __init__(self, direccion: str, metros: float, precio: float, habitaciones: int):
        if not direccion or not str(direccion).strip():
            raise ValueError("dirección obligatoria")
        if metros <= 0 or precio <= 0 or habitaciones < 1:
            raise ValueError("metros/precio > 0 y habitaciones >= 1")
        self._direccion = direccion.strip()
        self._metros = float(metros)
        self._precio = float(precio)
        self._habitaciones = int(habitaciones)
        self._estado = EstadoCasa.DISPONIBLE
        self._inquilino = None
        Vivienda.total_viviendas += 1

    @property
    def direccion(self):
        return self._direccion

    @property
    def precio(self):
        return self._precio

    @precio.setter
    def precio(self, valor):
        if not isinstance(valor, (int, float)) or isinstance(valor, bool) or valor <= 0:
            raise ValueError("precio debe ser > 0")
        self._precio = float(valor)

    @property
    def estado(self):
        return self._estado

    @property
    def metros(self):
        return self._metros

    def precio_m2(self) -> float:
        return self._precio / self._metros

    @classmethod
    def cambiar_impuesto_aitp(cls, nuevo: float) -> None:
        if not 0 <= nuevo <= 1:
            raise ValueError("AITP entre 0 y 1")
        cls.impuesto_aitp = nuevo

    @staticmethod
    def comparar_inversion(a: "Vivienda", b: "Vivienda") -> str:
        if a.precio_m2() < b.precio_m2():
            return f"Mejor €/m²: {a.direccion}"
        if b.precio_m2() < a.precio_m2():
            return f"Mejor €/m²: {b.direccion}"
        return "Mismo €/m²"

    def comprar(self) -> float:
        if self._estado != EstadoCasa.DISPONIBLE:
            raise ValueError("solo se compra si está DISPONIBLE")
        total = self._precio * (1 + self.impuesto_aitp)
        self._estado = EstadoCasa.VENDIDA
        return total

    def ficha(self) -> str:
        return (
            f"{self._direccion} | {self._metros} m² | {self._precio:,.0f} € | "
            f"{self._habitaciones} hab. | {self._estado.value}"
        )


v1 = Vivienda("Calle Mayor 1", 80, 200_000, 3)
v2 = Vivienda("Av. Libertad 10", 100, 220_000, 4)
print(v1.ficha())
print("Total viviendas creadas:", Vivienda.total_viviendas)
print("Coste compra c/AITP:", f"{v1.comprar():,.2f} €", "→", v1.estado)
# nueva disponible para comparar
v3 = Vivienda("Calle Sol 5", 70, 175_000, 2)
print(Vivienda.comparar_inversion(v2, v3))
print("€/m² v2:", round(v2.precio_m2(), 2), "| v3:", round(v3.precio_m2(), 2))

Calle Mayor 1 | 80.0 m² | 200,000 € | 3 hab. | disponible
Total viviendas creadas: 2
Coste compra c/AITP: 220,000.00 € → EstadoCasa.VENDIDA
Mejor €/m²: Av. Libertad 10
€/m² v2: 2200.0 | v3: 2500.0


## B2. Inquilino y regla del 35 %

La renta (nueva + ya comprometida) no puede superar el 35 % de los ingresos mensuales.


In [6]:
class Inquilino:
    RATIO_MAX = 0.35

    def __init__(self, nombre: str, dni: str, ingresos: float):
        if not nombre.strip():
            raise ValueError("nombre obligatorio")
        if ingresos <= 0:
            raise ValueError("ingresos > 0")
        self.nombre = nombre.strip()
        self._dni = dni.strip().upper()
        self._ingresos = float(ingresos)

    @property
    def dni(self):
        return self._dni

    @property
    def ingresos(self):
        return self._ingresos

    def puede_alquilar(self, renta: float, renta_ya_comprometida: float = 0.0) -> bool:
        if renta <= 0:
            raise ValueError("renta > 0")
        total = renta + renta_ya_comprometida
        return total <= self._ingresos * self.RATIO_MAX

    def techo_renta(self, renta_ya_comprometida: float = 0.0) -> float:
        return self._ingresos * self.RATIO_MAX - renta_ya_comprometida


ana = Inquilino("Ana López", "12345678Z", 3_000)
print("¿Puede 900 €?", ana.puede_alquilar(900))
print("¿Puede 900 + 400 ya comprometidos?", ana.puede_alquilar(900, 400))
print("Techo disponible:", ana.techo_renta(400), "€")

¿Puede 900 €? True
¿Puede 900 + 400 ya comprometidos? False
Techo disponible: 650.0 €


## B3. Alquiler: colaboración `Vivienda` ↔ `Inquilino`


In [7]:
def alquilar(vivienda: Vivienda, inquilino: Inquilino, renta: float, renta_comprometida: float = 0.0) -> str:
    if vivienda.estado != EstadoCasa.DISPONIBLE:
        raise ValueError("la vivienda no está disponible")
    if not inquilino.puede_alquilar(renta, renta_comprometida):
        raise ValueError(
            f"{inquilino.nombre} no cumple la regla del 35 % "
            f"(techo {inquilino.techo_renta(renta_comprometida):.0f} €)"
        )
    vivienda._estado = EstadoCasa.ALQUILADA
    vivienda._inquilino = inquilino
    vivienda._renta = renta
    return (
        f"Contrato OK: {vivienda.direccion} → {inquilino.nombre} "
        f"(DNI {inquilino.dni}) por {renta} €/mes"
    )


piso = Vivienda("Plaza España 3", 65, 180_000, 2)
print(alquilar(piso, ana, 900))
print(piso.ficha(), "| inquilino:", piso._inquilino.nombre)

# Segundo intento debe fallar: ya alquilada
try:
    alquilar(piso, Inquilino("Luis", "87654321X", 4_000), 800)
except ValueError as e:
    print("OK bloqueado:", e)

Contrato OK: Plaza España 3 → Ana López (DNI 12345678Z) por 900 €/mes
Plaza España 3 | 65.0 m² | 180,000 € | 2 hab. | alquilada | inquilino: Ana López
OK bloqueado: la vivienda no está disponible


## B4. Mini cartera: informe de estado


In [8]:
class GestionInmobiliaria:
    def __init__(self):
        self._propiedades: list[Vivienda] = []

    def agregar(self, vivienda: Vivienda) -> None:
        self._propiedades.append(vivienda)

    def disponibles(self) -> list[Vivienda]:
        return [v for v in self._propiedades if v.estado == EstadoCasa.DISPONIBLE]

    def informe(self) -> str:
        lineas = ["=== INFORME CARTERA ==="]
        for estado in EstadoCasa:
            grupo = [v for v in self._propiedades if v.estado == estado]
            lineas.append(f"{estado.value.upper()}: {len(grupo)}")
            for v in grupo:
                lineas.append("  - " + v.ficha())
        return "\n".join(lineas)


gestion = GestionInmobiliaria()
for v in (v2, v3, piso, Vivienda("Calle Luna 8", 90, 210_000, 3)):
    gestion.agregar(v)
print(gestion.informe())
print("Disponibles:", [v.direccion for v in gestion.disponibles()])

=== INFORME CARTERA ===
DISPONIBLE: 3
  - Av. Libertad 10 | 100.0 m² | 220,000 € | 4 hab. | disponible
  - Calle Sol 5 | 70.0 m² | 175,000 € | 2 hab. | disponible
  - Calle Luna 8 | 90.0 m² | 210,000 € | 3 hab. | disponible
RESERVADA: 0
VENDIDA: 0
ALQUILADA: 1
  - Plaza España 3 | 65.0 m² | 180,000 € | 2 hab. | alquilada
Disponibles: ['Av. Libertad 10', 'Calle Sol 5', 'Calle Luna 8']


### Tu turno B

**Ejercicio B1.** Añade el método de instancia `reservar(cliente: str)` que pase de `DISPONIBLE` → `RESERVADA`, y `liberar_reserva()` al revés. Prueba el ciclo.

**Ejercicio B2.** Implementa `rentabilidad_bruta(renta_mensual)` = `(renta × 12 / precio) × 100` en `Vivienda`.

**Ejercicio B3.** Cambia el AITP al 12 % con el método de clase y recalcula el coste de compra de una vivienda nueva.


In [9]:
# Escribe tu solución aquí (reserva, rentabilidad, AITP)


---
# Cierre — mapa hacia los repos completos

| Lo visto aquí | En el repo completo |
|---------------|---------------------|
| `Animal` / `Perro` / `Gato` | `src/modelos/` + tests + API opcional Swagger |
| `emitir_sonido`, `listar_info` | `src/servicios/gestion_animales.py` |
| `Vivienda`, AITP, estados | `src/modelos/casa.py`, enums en `utils/constantes.py` |
| Regla 35 % | `Inquilino` + contratos / UI agente-cliente |
| `GestionInmobiliaria` | servicio + persistencia SQLite + tkinter |

**Siguiente paso recomendado**

```bash
git clone https://github.com/gracobjo/poo_animales.git
git clone https://github.com/gracobjo/poo_inmobiliaria.git
```

Ejecuta `python main.py` y `python -m unittest discover -s tests -v` en cada uno.
